# Assignment 3 — Fashion-MNIST Classification

**IE500618 Machine Learning · Individual assignment**

This notebook builds and compares four small neural networks that classify images of clothing.

| Section | Content |
|---|---|
| 0. Setup | Imports and fixed settings used by every model |
| 1. Dataset and preparation | Load, inspect, normalize and split the data |
| Part A — Task 1 | Baseline model |
| Part A — Task 2 | Three experiments, one change each |
| Part A — Task 3 | Results table, model selection and test accuracy |

## 0. Setup

All settings that must be the same for every model are defined once here, so no experiment can differ by accident.

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

SEED = 67          # same seed for the data split and for every model's starting weights
EPOCHS = 10        # given in the assignment
BATCH_SIZE = 64    # given in the assignment

CLASS_NAMES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

## 1. Dataset and preparation

*Use the Fashion-MNIST dataset (`tf.keras.datasets.fashion_mnist.load_data()`), containing 28 × 28 grayscale images in 10 clothing categories. Normalize the pixel values to 0–1 and flatten each image into 784 input features (or use a Keras Flatten layer).*

### 1.1 Load the data
Keras downloads the dataset the first time and stores it locally. It comes in two parts:

- **X** = the images. This is what the model sees.
- **y** = the correct answers (labels), one number 0–9 per image, **in the same order**. `y_train_full[3]` is the answer for `X_train_full[3]`.

The shape `(60000, 28, 28)` means 60 000 different images, each 28 rows × 28 columns of pixels.

In [ ]:
(X_train_full, y_train_full), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

print("Training images:", X_train_full.shape)   # (number of images, rows, columns)
print("Training labels:", y_train_full.shape)
print("Test images:    ", X_test.shape)
print("Test labels:    ", y_test.shape)

### 1.2 Look at the data
The first 25 images. The title is the label from `y`: the correct answer, not a guess. No model has been built yet.

Note the difference between **position** and **class**: image at position 0 is an ankle boot, which is class 9. The images are mixed, not sorted by class.

Each pixel is a number from 0 (black) to 255 (white). Row 15 of the boot shows zeros for the background and high values where the boot is.

In [ ]:
plt.figure(figsize=(9, 9))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.imshow(X_train_full[i], cmap="gray")
    plt.title(f"{y_train_full[i]}: {CLASS_NAMES[y_train_full[i]]}", fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.show()

print("Row 15 of image 0 (raw pixel values 0–255):")
print(X_train_full[0][15])

### 1.3 Normalize the pixel values to 0–1
Dividing by 255 gives 0 = black and 1 = white. The image looks the same, but small inputs make training more stable.

In [ ]:
X_train_full = X_train_full / 255.0
X_test = X_test / 255.0

print("Row 15 of image 0 after normalization:")
print(X_train_full[0][15].round(2))

### 1.4 Split off a validation set
Three sets, each with images (X) and answers (y):

```
60 000 training images ──┬── 48 000  X_train, y_train   learn from these     (exercises)
                         └── 12 000  X_val,   y_val     compare the models   (mock exam)
10 000 test images ──────────────── X_test,  y_test    used once at the end (real exam)
```

Choosing the model on the test set would be like seeing the exam in advance, so the test set is not touched until Task 3.

- `stratify`: same share of each class in both parts
- `random_state=SEED`: same split every run

Flattening (28 × 28 → one row of 784 numbers) happens later, in a `Flatten` layer inside each model.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2,
    stratify=y_train_full,
    random_state=SEED
)

print(f"Training:   {X_train.shape[0]:>6} images")
print(f"Validation: {X_val.shape[0]:>6} images")
print(f"Test:       {X_test.shape[0]:>6} images")

---
## Part A — Build and compare neural networks

### Helper function
All four models are built the same way. Only three settings change, so the recipe is written once and each model is one call.

The function does four things:

1. **Build** the network: `28 × 28 image → Flatten (784) → hidden layer → 10 outputs (softmax, one per class)`
2. **Compile**: choose the optimizer (how the weights are adjusted), the loss (how wrong a guess is) and the metric (accuracy)
3. **Fit** (train): 10 epochs, batch size 64. 48 000 / 64 = 750 weight updates per epoch
4. **Record** the final training and validation accuracy

Reading the output, one line per epoch:
- `accuracy`, `loss`: measured on the training images
- `val_accuracy`, `val_loss`: the same, measured on the validation images, which the model does not learn from

The seed is reset first, so every model starts from the same random weights. That keeps the comparison fair.

In [ ]:
results = {}   # filled with one entry per model

def build_and_train(name, hidden_neurons, activation, optimizer):
    tf.keras.utils.set_random_seed(SEED)       # same starting weights for every model

    # 1. Build
    model = tf.keras.Sequential([
        tf.keras.Input(shape=(28, 28)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(hidden_neurons, activation=activation),
        tf.keras.layers.Dense(10, activation="softmax")
    ], name=name.replace(" ", "_"))

    # 2. Compile
    model.compile(
        optimizer=optimizer,            # default Keras settings
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    # 3. Fit (train)
    print(f"=== {name}: {hidden_neurons} neurons, {activation}, {optimizer} ===")
    history = model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_val, y_val),
        verbose=2                       # one line per epoch
    )

    # 4. Record the result of the last epoch
    train_acc = history.history["accuracy"][-1]
    val_acc = history.history["val_accuracy"][-1]
    print(f"\n>>> {name} — train accuracy: {train_acc:.4f} | validation accuracy: {val_acc:.4f}\n")

    results[name] = {
        "model": model, "history": history,
        "neurons": hidden_neurons, "activation": activation, "optimizer": optimizer,
        "train_acc": train_acc, "val_acc": val_acc,
    }
    return model

---
### Task 1 — Baseline
*Build a baseline model with one hidden layer of 32 neurons using ReLU activation and the Adam optimizer. Train it and record its training and validation accuracy.*

The baseline is the starting point. Each experiment is compared against it.

`summary()` shows the layers and the number of trainable parameters (weights and biases) in each.

In [ ]:
baseline = build_and_train("Baseline", hidden_neurons=32, activation="relu", optimizer="adam")
baseline.summary()

### Task 2 — Three experiments
*Starting from the baseline, run three more experiments. For each experiment, change only the setting shown below; keep the other settings the same as the baseline.*

| Model | Hidden neurons | Hidden activation | Optimizer |
|---|---|---|---|
| Baseline | 32 | ReLU | Adam |
| Experiment 1 | **64** | ReLU | Adam |
| Experiment 2 | 32 | **tanh** | Adam |
| Experiment 3 | 32 | ReLU | **SGD** |

#### Experiment 1 — 64 hidden neurons
More neurons = a bigger network that can learn more patterns. Does it improve **validation** accuracy, or only training accuracy?

In [ ]:
exp1 = build_and_train("Experiment 1", hidden_neurons=64, activation="relu", optimizer="adam")

#### Experiment 2 — tanh activation
ReLU turns negative values into 0. tanh squeezes every value into the range −1 to 1.

In [ ]:
exp2 = build_and_train("Experiment 2", hidden_neurons=32, activation="tanh", optimizer="adam")

#### Experiment 3 — SGD optimizer
Adam adapts the step size for each weight. SGD takes equal-sized steps (default learning rate 0.01).

In [ ]:
exp3 = build_and_train("Experiment 3", hidden_neurons=32, activation="relu", optimizer="sgd")

### Task 3 — Results, model selection and test accuracy
*Record the training and validation accuracy of all four models in a results table. Identify the model with the highest validation accuracy. Evaluate that model on the test set and report its test accuracy. Do not use the test set to choose the model.*

#### Results table

In [ ]:
print(f"{'Model':<14}{'Neurons':>8}  {'Activation':<11}{'Optimizer':<10}{'Train acc':>10}{'Val acc':>10}")
print("-" * 63)
for name, r in results.items():
    print(f"{name:<14}{r['neurons']:>8}  {r['activation']:<11}{r['optimizer']:<10}"
          f"{r['train_acc']:>10.4f}{r['val_acc']:>10.4f}")

#### Validation accuracy per epoch
The curves show how each model improved during training, not only where it ended.

In [ ]:
plt.figure(figsize=(8, 5))
for name, r in results.items():
    plt.plot(range(1, EPOCHS + 1), r["history"].history["val_accuracy"], marker="o", label=name)
plt.xlabel("Epoch")
plt.ylabel("Validation accuracy")
plt.title("Validation accuracy per epoch")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

#### Select the best model
The model is chosen **only** by validation accuracy. The test set has not been used up to this point.

In [ ]:
best_name = max(results, key=lambda name: results[name]["val_acc"])
best_model = results[best_name]["model"]

print(f"Highest validation accuracy: {best_name} ({results[best_name]['val_acc']:.4f})")

#### Test accuracy of the selected model
The test set is used once, to give an honest estimate of how the chosen model performs on images it has never seen.

In [ ]:
test_loss, test_acc = best_model.evaluate(X_test, y_test, verbose=0)

print(f"Selected model: {best_name}")
print(f"Test accuracy:  {test_acc:.4f}")
print(f"Test loss:      {test_loss:.4f}")